In [1]:
# Dataset citation: Brazilian University Paper Abstracts Dataset (2013–2023), Classified by Sustainable Development Goals (SDGs), supplied files in the correspondingly named folder. The folder contains original, preprocessed, balanced, training, and testing CSVs plus The_files_and_their_contents.pdf; no authorship or publication citation was listed in a text README.

In [ ]:
# Imports and simple EDA helpers
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

DATA_ROOT = Path("..")

def inspect_table(df, name, preview_columns=None):
    print(f"\n{name}: {len(df):,} rows x {len(df.columns):,} columns")
    cols = preview_columns or list(df.columns[:8])
    shown = df[cols]
    display(pd.DataFrame({"column": shown.columns, "dtype": [str(x) for x in shown.dtypes],
                          "missing": shown.isna().sum().values,
                          "missing_%": (shown.isna().mean().values * 100).round(2),
                          "unique": shown.astype(str).nunique(dropna=True).values}))
    print(f"Duplicate rows on shown columns: {shown.astype(str).duplicated().sum():,}")
    display(shown.head(5))

def plot_counts(counts, title, xlabel="Class", ylabel="Rows"):
    counts = counts.sort_index()
    display(counts.rename("count").to_frame())
    ax = counts.plot(kind="bar", figsize=(11, 4), color="#4472C4")
    ax.set(title=title, xlabel=xlabel, ylabel=ylabel)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

def text_stats(df, column):
    if column not in df.columns:
        return
    values = df[column].fillna("").astype(str)
    lengths = values.str.len()
    words = values.str.split().str.len()
    print(f"\nText field: {column}")
    display(pd.DataFrame({"characters": lengths.describe(), "words": words.describe()}))
    print("Empty text rows:", int(values.str.strip().eq("").sum()))

In [ ]:
# Load files
folder = DATA_ROOT / "Brazilian University Paper Abstracts Dataset (2013-2023), Classified by Sustainable Development Goals (SDGs)"
file_names = ["original_dataset.csv", "original_training_set.csv", "original_testing_set.csv", "preprocessed_dataset.csv", "balanced_training_set.csv"]

def read_brazilian_csv(path):
    # Abstract text can contain semicolons, so split from the right.
    lines = path.read_text(encoding="utf-8-sig").splitlines()
    header = lines[0].split(";")
    rows = [line.rsplit(";", len(header) - 1) for line in lines[1:] if line.strip()]
    return pd.DataFrame(rows, columns=header)

frames = {name.removesuffix(".csv"): read_brazilian_csv(folder / name) for name in file_names}

In [ ]:
# File row counts
display(pd.DataFrame([{"file": name, "rows": len(frame), "columns": len(frame.columns)} for name, frame in frames.items()]))

,file,rows,columns
0,original_dataset,64016,20
1,original_training_set,11030,2
2,original_testing_set,2759,2
3,preprocessed_dataset,13789,2
4,balanced_training_set,5100,3


In [ ]:
# Original dataset table
original = frames["original_dataset"]
inspect_table(original, "Original dataset", list(original.columns))


Original dataset: 64,016 rows x 20 columns


,column,dtype,missing,missing_%,unique
0,abstract_text,object,0,0.00,25653
1,Sustainable Development Goals (2023),object,45147,70.52,2913
2,SDG 10,object,46997,73.41,1323
3,SDG 11,object,47691,74.50,739
4,SDG 12,object,47992,74.97,476
5,SDG 13,object,48198,75.29,290
6,SDG 14,object,48318,75.48,187
7,SDG 15,object,48397,75.60,118
8,SDG 16,object,48425,75.65,95
9,SDG 17,object,48456,75.69,66


Duplicate rows on shown columns: 38,307


,abstract_text,Sustainable Development Goals (2023),SDG 10,SDG 11,SDG 12,SDG 13,SDG 14,SDG 15,SDG 16,SDG 17,SDG 2,SDG 3,SDG 4,SDG 5,SDG 6,SDG 7,SDG 8,SDG 9,SDG 1,label
0,"""",None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
1,In this manuscript we report...,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
2,x,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
3,@Pt,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
4,y,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None


In [ ]:
# Training table
train = frames["original_training_set"]
inspect_table(train, "Original training set", list(train.columns))


Original training set: 11,030 rows x 2 columns


,column,dtype,missing,missing_%,unique
0,abstract,object,0,0.0,11030
1,label,object,0,0.0,17


Duplicate rows on shown columns: 0


,abstract,label
0,Background We investigated the biomarkers of i...,3
1,"Since November of 2015, when ore tailings from...",14
2,"""Aims We investigated the effects of physical ...",3
3,"""Over the last few decades, several emerging o...",3
4,"""Background A low supply of donated organs led...",3


In [ ]:
# Test table
test = frames["original_testing_set"]
inspect_table(test, "Original testing set", list(test.columns))


Original testing set: 2,759 rows x 2 columns


,column,dtype,missing,missing_%,unique
0,abstract,object,0,0.0,2759
1,label,object,0,0.0,17


Duplicate rows on shown columns: 0


,abstract,label
0,One of the hurdles of renewable energy product...,7
1,Hepatitis B virus (HBV) is distributed worldwi...,3
2,The Parnaíba basin is located in the Northeast...,17
3,"""Biological invasions need to be efficiently m...",14
4,"""The bioprospection of zootherapeutic products...",15


In [ ]:
# Preprocessed table
inspect_table(frames["preprocessed_dataset"], "Preprocessed dataset", list(frames["preprocessed_dataset"].columns))


Preprocessed dataset: 13,789 rows x 2 columns


,column,dtype,missing,missing_%,unique
0,preprocessed_abstract_text,object,0,0.0,13789
1,label,object,0,0.0,17


Duplicate rows on shown columns: 0


,preprocessed_abstract_text,label
0,manuscript report facile synthesis route elect...,7
1,cystoseira sargassaceae genus marine brown alg...,3
2,colorectal cancer one common types cancer acco...,3
3,aims cardiac arrhythmias one important remote ...,3
4,hesperidin abundant flavanone cheap byproduct ...,3


In [ ]:
# Balanced training table
inspect_table(frames["balanced_training_set"], "Balanced training set", list(frames["balanced_training_set"].columns))


Balanced training set: 5,100 rows x 3 columns


,column,dtype,missing,missing_%,unique
0,abstract,object,0,0.0,5090
1,label,object,0,0.0,17
2,synthetic,object,0,0.0,2


Duplicate rows on shown columns: 6


,abstract,label,synthetic
0,Background aims Irisin involved compensatory m...,3,False
1,Background The role adjuvant chemotherapy bili...,3,False
2,Differently two dimensional cell culture three...,3,False
3,Staphylococcus aureus Gram positive bacterium ...,3,False
4,An insult trigger protective response even cel...,3,False


In [ ]:
# Train and test class counts
display(pd.concat({"train": train["label"].value_counts(), "test": test["label"].value_counts()}, axis=1).fillna(0).astype(int).sort_index())

,train,test
label,,
1,2,1
10,26,7
11,123,31
12,70,18
13,138,35
14,320,80
15,234,59
16,45,11
17,1896,474


In [ ]:
# Balanced training class counts
display(frames["balanced_training_set"]["label"].value_counts().sort_index().rename_axis("SDG label").to_frame("rows"))

,rows
SDG label,
1,300
10,300
11,300
12,300
13,300
14,300
15,300
16,300
17,300


In [ ]:
# Abstract length
text_stats(original, "abstract")
text_stats(train, "abstract")
text_stats(test, "abstract")


Text field: abstract


,characters,words
count,11030.000000,11030.000000
mean,1517.061831,221.106346
std,495.209171,73.796700
min,218.000000,28.000000
25%,1201.000000,173.000000
50%,1516.500000,220.000000
75%,1774.000000,256.000000
max,6764.000000,955.000000


Empty text rows: 0

Text field: abstract


,characters,words
count,2759.000000,2759.000000
mean,1512.544038,220.271113
std,478.331424,71.154153
min,211.000000,25.000000
25%,1199.000000,174.000000
50%,1513.000000,220.000000
75%,1768.000000,255.000000
max,4405.000000,641.000000


Empty text rows: 0
